# Custom LLM Training for Self-Learning Chatbot (Cloud Run)

Cloud-optimized training pipeline for fine-tuning **Llama 3.1 8B** using **QLoRA** for the SmartSelf AI project.

## Objectives
- Run reliably on Colab/Kaggle/cloud GPUs
- Use memory-efficient 4-bit QLoRA setup
- Enable resume from checkpoints and periodic saves
- Include batch inference and ChromaDB RAG integration

In [ ]:
# Cloud dependency setup (single command for reliability)
!pip -q install -U "transformers>=4.43.0" "peft>=0.11.0" "bitsandbytes>=0.43.1" "accelerate>=0.31.0" "datasets>=2.20.0" "wandb>=0.17.0" "sentence-transformers>=3.0.1" "chromadb>=0.5.0" "matplotlib>=3.8.0" "scikit-learn>=1.4.0"

In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import wandb

from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Imports complete. Seed:", SEED)

In [ ]:
# Cloud-friendly config
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B"
OUTPUT_DIR = "./cloud_custom_llm_model"
DATA_PATH = "./data/training_data.json"
VAL_SPLIT = 0.1
MAX_LENGTH = 1024  # reduced from 2048 for better cloud memory usage

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
WARMUP_STEPS = 50
MAX_STEPS = -1
SAVE_STEPS = 200
EVAL_STEPS = 200

# Mixed precision auto-selection for cloud GPUs
USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
USE_FP16 = not USE_BF16
USE_4BIT = True

print("BF16:", USE_BF16, "| FP16:", USE_FP16, "| 4-bit:", USE_4BIT)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# GPU and memory check
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(idx)
    total_gb = props.total_memory / (1024 ** 3)
    reserved_gb = torch.cuda.memory_reserved(idx) / (1024 ** 3)
    allocated_gb = torch.cuda.memory_allocated(idx) / (1024 ** 3)
    print(f"GPU: {props.name}")
    print(f"Total VRAM: {total_gb:.2f} GB | Reserved: {reserved_gb:.2f} GB | Allocated: {allocated_gb:.2f} GB")
else:
    print("No GPU detected. Enable GPU runtime in your cloud environment.")

In [ ]:
# Optional experiment tracking
WANDB_DISABLED = True  # set False if you want tracking
if not WANDB_DISABLED:
    wandb.init(
        project="smartself-llm-cloud",
        name="llama31-8b-qlora-cloud",
        config={"model": MODEL_NAME, "lr": LEARNING_RATE, "batch_size": BATCH_SIZE, "max_length": MAX_LENGTH},
    )
    print("wandb initialized")
else:
    print("wandb disabled")

In [ ]:
# Data utilities + sample data fallback
def create_sample_data_if_missing(path):
    if os.path.exists(path):
        print("Training data found:", path)
        return

    print("Training data missing, creating sample dataset...")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    sample = [
        {"instruction": "What is machine learning?", "context": "AI subfield", "response": "Machine learning enables systems to learn from data."},
        {"instruction": "Explain deep learning.", "context": "Uses neural networks", "response": "Deep learning uses multilayer neural networks for complex representations."},
        {"instruction": "What is NLP?", "context": "Language + AI", "response": "NLP helps computers understand and generate human language."},
        {"instruction": "What is RAG?", "context": "Retrieval + generation", "response": "RAG combines retrieval from a knowledge base with language generation."},
    ]
    with open(path, "w", encoding="utf-8") as f:
        json.dump(sample, f, ensure_ascii=False, indent=2)
    print("Sample data written to", path)

def format_instruction(ex):
    return f"""### Instruction:\n{ex.get('instruction','')}\n\n### Context:\n{ex.get('context','')}\n\n### Response:\n{ex.get('response','')}"""

create_sample_data_if_missing(DATA_PATH)
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

formatted = [format_instruction(x) for x in raw_data]
train_data, val_data = train_test_split(formatted, test_size=VAL_SPLIT, random_state=SEED)
train_dataset = Dataset.from_dict({"text": train_data})
val_dataset = Dataset.from_dict({"text": val_data})

print(f"Loaded {len(raw_data)} examples | Train: {len(train_dataset)} | Val: {len(val_dataset)}")

In [ ]:
# Tokenizer + tokenization
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

def tokenize_batch(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

tokenized_train = train_dataset.map(tokenize_batch, batched=True, remove_columns=["text"], desc="Tokenizing train")
tokenized_val = val_dataset.map(tokenize_batch, batched=True, remove_columns=["text"], desc="Tokenizing val")

print("Tokenization complete.", len(tokenized_train), len(tokenized_val))

In [ ]:
# Load quantized base model + apply LoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=USE_4BIT,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config if USE_4BIT else None,
    device_map="auto",
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Trainer setup
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    max_steps=MAX_STEPS,
    logging_steps=10,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=USE_FP16,
    bf16=USE_BF16,
    optim="paged_adamw_32bit",
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    report_to=[] if WANDB_DISABLED else ["wandb"],
    save_total_limit=2,
    seed=SEED,
    data_seed=SEED,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer,
)
print("Trainer initialized")

In [ ]:
# Resume training helper (set path or keep None)
RESUME_CHECKPOINT = None  # e.g., "./cloud_custom_llm_model/checkpoint-200"

print("Starting training...")
if RESUME_CHECKPOINT:
    print("Resuming from:", RESUME_CHECKPOINT)
    trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT)
else:
    trainer.train()
print("Training complete")

In [ ]:
# Plot training/eval metrics
logs = trainer.state.log_history
train_losses = [x["loss"] for x in logs if "loss" in x]
train_steps = [x["step"] for x in logs if "loss" in x]
eval_losses = [x["eval_loss"] for x in logs if "eval_loss" in x]
eval_steps = [x["step"] for x in logs if "eval_loss" in x]

if train_losses:
    plt.figure(figsize=(10, 4))
    plt.plot(train_steps, train_losses, label="train_loss")
    if eval_losses:
        plt.plot(eval_steps, eval_losses, label="eval_loss")
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.title("Training Curves")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()
else:
    print("No logged losses found.")

In [ ]:
# Save model and adapters
final_model_path = os.path.join(OUTPUT_DIR, "final_model")
lora_path = os.path.join(OUTPUT_DIR, "lora_adapters")

trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
model.save_pretrained(lora_path)

print("Saved final model:", final_model_path)
print("Saved LoRA adapters:", lora_path)

In [ ]:
# Inference helpers (single + batch)
def generate_response(prompt, max_new_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

tests = [
    {"instruction": "What is machine learning?", "context": "AI", "response": ""},
    {"instruction": "Explain RAG briefly.", "context": "retrieval", "response": ""},
    {"instruction": "How does LoRA help?", "context": "parameter-efficient tuning", "response": ""},
]

for i, t in enumerate(tests, 1):
    prompt = f"### Instruction:\n{t['instruction']}\n\n### Context:\n{t['context']}\n\n### Response:\n"
    print("\n" + "=" * 50)
    print(f"Test {i}: {t['instruction']}")
    print(generate_response(prompt, max_new_tokens=120))

In [ ]:
# ChromaDB integration example for RAG
from chromadb import Client
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

chroma = Client(Settings(anonymized_telemetry=False, allow_reset=True))
embedder = SentenceTransformer("all-MiniLM-L6-v2")

name = "smartself_cloud_kb"
try:
    collection = chroma.get_collection(name=name)
except Exception:
    collection = chroma.create_collection(name=name)

docs = [
    "QLoRA enables low-memory fine-tuning using quantization and adapters.",
    "RAG augments generation by retrieving relevant context from a vector database.",
    "ChromaDB stores embeddings and supports semantic retrieval for chatbot context.",
]
ids = [f"cloud_doc_{i}" for i in range(len(docs))]

try:
    collection.upsert(
        documents=docs,
        ids=ids,
        metadatas=[{"source": i} for i in ids],
    )
except Exception:
    pass

query = "Why use QLoRA for cloud training?"
q_emb = embedder.encode(query).tolist()
res = collection.query(query_embeddings=[q_emb], n_results=2)
ctx = "\n".join(res["documents"][0]) if res.get("documents") else ""

prompt = f"### Instruction:\n{query}\n\n### Context:\n{ctx}\n\n### Response:\n"
print("Retrieved context:\n", ctx)
print("\nRAG output:\n", generate_response(prompt, max_new_tokens=140))

In [ ]:
# Cleanup + summary
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=" * 60)
print("CLOUD TRAINING SUMMARY")
print("=" * 60)
print("Model:", MODEL_NAME)
print("Output:", OUTPUT_DIR)
print("Epochs:", NUM_EPOCHS)
print("Batch x GradAcc:", BATCH_SIZE, "x", GRADIENT_ACCUMULATION_STEPS)
if 'train_losses' in globals() and train_losses:
    print("Final train loss:", round(train_losses[-1], 4))
if 'eval_losses' in globals() and eval_losses:
    print("Final eval loss:", round(eval_losses[-1], 4))

if not WANDB_DISABLED:
    wandb.finish()

print("Done.")